# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


Finding 1 - "What Predicts Health?" (Random Forest feature importance, p.27)
The paper defines Health Score as (Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts).
The Random Forest's top predictors of that same score are Position (43%), Impressions (32%), Scroll Depth (15%), and CTR (8%) - the exact four components the score is built from. Does the validation design carry the claim? No — the paper says the model is "holdout-tested," but a holdout split doesn't rescue this: if the label is a weighted sum of the features, a model will score well on any holdout by re-deriving the same formula, not by learning anything new. High out-of-sample accuracy here proves the arithmetic is consistent, not that these are real drivers of quality. The paper does flag this itself ("importance is descriptive rather than causal") — a good instinct — but a reader skimming the bar chart could still walk away thinking "improve position and impressions to raise health score," which is circular advice: those ARE the score. This is the same shortcut my own `w05` decision tree took with `imp_late` — a model finding the label's own construction instead of a genuine pattern.

**Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.29)**
Two questions the write-up doesn't answer. First: what's the base rate? 71% accuracy on a "growing vs declining" split only means something next to the naive floor — if roughly 70% of sampled pages are naturally growing (plausible, given the report's own headline shows +70.6% impressions 30-day trend), 71% could be barely above predicting the majority class every time — the same check I ran against `base_rate` all through `w05`. Second: the methodology page describes every model in the ML pipeline as an "80/20 split" with no mention of grouping by brand — but this dataset spans 57 brands. A random row-wise split risks the same client-leakage problem that pushed me toward a size-balanced GroupKFold in `w05`: a model could partly be learning "this brand's typical content" rather than a real growth signal, and a random split wouldn't catch that.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


`w05` already used a size-balanced GroupKFold split — that IS the honest version. So the "before" here is the naive version: a plain random 80/20 row-wise split, ignoring `client_hash_id` entirely (the same kind of split the FlyRank paper's methodology page describes for its own models). If the naive split scores noticeably higher than the grouped OOF scores from `w05`, that gap IS the client-leakage effect — a concrete, numeric answer to the methodology question raised in Section 1.

In [4]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
DECISION_MOMENT = "2026-04-30"
WINDOW_START = "2026-01-30"
SPLIT_DATE = "2026-03-15"

#read the daily content performance data from Parquet files covering January through April 2026
#filter it to the specific feature window between WINDOW_START and DECISION_MOMENT
#group the data by client and page
#split impressions into an early period (imp_early) and a later period (imp_late) based on SPLIT_DATE
#calculates total impressions, total clicks, and average search position for the entire window
#produce a DataFrame with one row per client-page combination

feature_paths = [f"{rel}/fact_content_daily_performance/month=2026-0{m}/*.parquet" for m in [1, 2, 3, 4]]

feature_df = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet([{', '.join(f"'{p}'" for p in feature_paths)}])
        WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{DECISION_MOMENT}'
    )


    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_early,
        SUM(CASE WHEN report_date >  DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_late,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

qualifying = con.sql(f"""
    SELECT c.content_hash_id, c.client_hash_id, c.content_type, c.main_intent,
           c.word_count, c.char_count, c.competition_level, c.search_volume
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/dim_clients.parquet') cl USING (client_hash_id)
    WHERE c.content_created_date <= DATE '{WINDOW_START}'
      AND cl.gsc_data_start <= DATE '{WINDOW_START}'
""").df()

feature_df = feature_df.merge(qualifying, on=["client_hash_id", "content_hash_id"], how="inner")
feature_df["ctr"] = np.where(
    feature_df["impressions_90d"] > 0,
    (feature_df["clicks_90d"] / feature_df["impressions_90d"]) * 100,
    np.nan,
)
feature_df["is_declining"] = (feature_df["imp_late"] < feature_df["imp_early"]).astype(int)

print(f"Rows: {len(feature_df):,}")
feature_df.head()

Token loaded: hf_IcB... (length 37)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285


,client_hash_id,content_hash_id,imp_early,imp_late,impressions_90d,clicks_90d,avg_position_90d,content_type,main_intent,word_count,char_count,competition_level,search_volume,ctr,is_declining
0,client_e547b89c05043229,content_7292af5ee9096ce1,1438.0,1802.0,3240.0,2.0,35.669767,keyword article,informational,1471,9089,LOW,50,0.061728,0
1,client_e547b89c05043229,content_14a6ade39e9a8e31,824.0,1770.0,2594.0,3.0,21.947559,keyword article,transactional,2996,18548,HIGH,20,0.115652,0
2,client_e547b89c05043229,content_5ed859ae9dc2e356,0.0,0.0,0.0,0.0,NaN,keyword article,transactional,1611,9874,LOW,10,NaN,0
3,client_e547b89c05043229,content_2618be372e19730f,400.0,709.0,1109.0,0.0,28.642764,keyword article,commercial,1497,9416,MEDIUM,10,0.000000,0
4,client_e547b89c05043229,content_1da7d268111c5875,1209.0,875.0,2084.0,3.0,19.449698,keyword article,transactional,2760,17157,LOW,10,0.143954,1


In [5]:
may_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_may
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-05/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

model_df = feature_df.merge(may_df, on=["client_hash_id", "content_hash_id"], how="left")
model_df["impressions_may"] = model_df["impressions_may"].fillna(0)

LATE_DAYS, MAY_DAYS = 46, 31
model_df["rate_late"] = model_df["imp_late"] / LATE_DAYS
model_df["rate_may"] = model_df["impressions_may"] / MAY_DAYS
model_df["still_declining_may"] = (model_df["rate_may"] < 0.8 * model_df["rate_late"]).astype(int)

print(f"Rows: {len(model_df):,}")
print(f"still_declining_may base rate: {model_df['still_declining_may'].mean():.3f}")

numeric_features = ["imp_early", "imp_late", "impressions_90d", "clicks_90d", "avg_position_90d", "ctr",
                     "word_count", "char_count", "search_volume"]
categorical_features = ["content_type", "main_intent", "competition_level"]
banned = {"is_declining", "impressions_may", "rate_late", "rate_may", "still_declining_may",
          "content_hash_id", "client_hash_id"}
assert not (set(numeric_features) | set(categorical_features)) & banned, "leakage/grouping column in feature list"
print(f"Numeric features: {len(numeric_features)}  |  Categorical features: {len(categorical_features)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285
still_declining_may base rate: 0.289
Numeric features: 9  |  Categorical features: 3


In [7]:
RANDOM_STATE = 42
N_FOLDS = 5

client_sizes = model_df.groupby("client_hash_id").size().sort_values(ascending=False)
fold_totals = np.zeros(N_FOLDS, dtype=int)
client_to_fold = {}
for client, size in client_sizes.items():
    smallest_fold = int(np.argmin(fold_totals))
    client_to_fold[client] = smallest_fold
    fold_totals[smallest_fold] += size

print("Rows per fold after size-balancing:")
for f in range(N_FOLDS):
    n_clients = sum(1 for v in client_to_fold.values() if v == f)
    print(f"  fold {f}: {fold_totals[f]:>7,} rows across {n_clients} clients")

model_df["fold"] = model_df["client_hash_id"].map(client_to_fold)

Rows per fold after size-balancing:
  fold 0:  52,005 rows across 7 clients
  fold 1:  52,065 rows across 9 clients
  fold 2:  52,165 rows across 7 clients
  fold 3:  51,998 rows across 9 clients
  fold 4:  52,052 rows across 9 clients


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

def build_matrix(frame, columns=None):
    num = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[categorical_features].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
    mat = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if columns is not None:
        mat = mat.reindex(columns=columns, fill_value=0)
    return mat

y = model_df["still_declining_may"].to_numpy()

model_builders = {
    "logistic_regression": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "decision_tree": lambda: DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=50,
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ),
}

oof_scores = {name: np.zeros(len(model_df)) for name in model_builders}
for f in range(N_FOLDS):
    train_mask = model_df["fold"] != f
    test_mask = model_df["fold"] == f
    X_train = build_matrix(model_df[train_mask])
    X_test = build_matrix(model_df[test_mask], columns=X_train.columns)
    y_train = y[train_mask.to_numpy()]
    for name, builder in model_builders.items():
        model = builder()
        model.fit(X_train, y_train)
        oof_scores[name][test_mask.to_numpy()] = model.predict_proba(X_test)[:, 1]
    print(f"fold {f}: train rows={train_mask.sum():,}  test rows={test_mask.sum():,}")

# baseline score, w04's exact rule, recomputed here
visible = model_df["impressions_90d"] >= 500
position_ok = (model_df["avg_position_90d"] > 0) & (model_df["avg_position_90d"] <= 20)
ctr_low = model_df["ctr"] < 0.4
declining = model_df["is_declining"] == 1
flag = visible & position_ok & ctr_low & declining
baseline_score = np.where(flag, model_df["impressions_90d"], 0)
print(f"\nBaseline flagged: {flag.sum():,} / {len(model_df):,}")

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

rows = []
for k in [20, 50, 100, 200]:
    row = {"k": k, "base_rate": round(y.mean(), 3), "baseline": round(precision_at_k(y, baseline_score, k), 3)}
    for name in model_builders:
        row[name] = round(precision_at_k(y, oof_scores[name], k), 3)
    rows.append(row)

comparison_table = pd.DataFrame(rows).set_index("k")
comparison_table

fold 0: train rows=208,280  test rows=52,005
fold 1: train rows=208,220  test rows=52,065
fold 2: train rows=208,120  test rows=52,165
fold 3: train rows=208,287  test rows=51,998
fold 4: train rows=208,233  test rows=52,052

Baseline flagged: 22,971 / 260,285


,base_rate,baseline,logistic_regression,decision_tree,random_forest
k,,,,,
20,0.289,0.60,0.30,0.800,0.900
50,0.289,0.62,0.42,0.760,0.800
100,0.289,0.61,0.43,0.720,0.840
200,0.289,0.62,0.41,0.735,0.835


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import numpy as np

def build_matrix(frame, columns=None):
    num = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[categorical_features].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
    mat = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if columns is not None:
        mat = mat.reindex(columns=columns, fill_value=0)
    return mat

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

y = model_df["still_declining_may"].to_numpy()

# NAIVE split: random 80/20, ignoring client_hash_id entirely
rng = np.random.default_rng(42)
shuffled_idx = rng.permutation(len(model_df))
split_point = int(len(model_df) * 0.8)
train_idx, test_idx = shuffled_idx[:split_point], shuffled_idx[split_point:]

train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_train = build_matrix(train_df)
X_test = build_matrix(test_df, columns=X_train.columns)
y_train, y_test = y[train_idx], y[test_idx]

naive_tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=42)
naive_tree.fit(X_train, y_train)
naive_scores = naive_tree.predict_proba(X_test)[:, 1]

print("Client overlap check — how much train/test overlap does the naive split allow?")
train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])
print(f"Clients in both train AND test: {len(train_clients & test_clients)} / {model_df['client_hash_id'].nunique()}")

print("\nBEFORE (naive 80/20, no grouping) vs AFTER (w05's size-balanced GroupKFold OOF):")
for k in [20, 50, 100, 200]:
    naive_p = precision_at_k(y_test, naive_scores, min(k, len(y_test)))
    honest_p = precision_at_k(y, oof_scores["decision_tree"], k)
    print(f"  k={k:>3}: naive={naive_p:.3f}   honest(grouped OOF)={honest_p:.3f}   gap={naive_p - honest_p:+.3f}")


Client overlap check — how much train/test overlap does the naive split allow?
Clients in both train AND test: 41 / 41

BEFORE (naive 80/20, no grouping) vs AFTER (w05's size-balanced GroupKFold OOF):
  k= 20: naive=0.850   honest(grouped OOF)=0.800   gap=+0.050
  k= 50: naive=0.880   honest(grouped OOF)=0.760   gap=+0.120
  k=100: naive=0.800   honest(grouped OOF)=0.720   gap=+0.080
  k=200: naive=0.790   honest(grouped OOF)=0.735   gap=+0.055


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*




Restating what `w03_feature_leakage_check.ipynb` already established, applied to the final feature set actually used in `w05`'s model:

- **Excluded, confirmed:** `content_updated_date`, `last_optimized_date` (snapshot-table dates that don't reliably reflect pre-decision-moment state — see `w03` for the full trap demonstration), `optimization_eligible_date` (forward-looking product field), `is_published`/`is_deleted` (product state, not a search signal), any `fact_content_daily_performance` row after `2026-04-30` (May/June — used only as the check-only label source, never a feature).
- **Included, and why each is safe:** `imp_early`, `imp_late`, `impressions_90d`, `clicks_90d`, `avg_position_90d`, `ctr` — all observed strictly within the `2026-01-30`–`2026-04-30` feature window. `content_type`, `main_intent`, `word_count`, `char_count`, `competition_level`, `search_volume` — `dim_content` metadata, stable content properties rather than time-sensitive state.
- **The one caveat this course of notebooks surfaced that's worth restating here:** even without any literal future-window or product-flag leakage, `w05`'s Section 4 found that `imp_late` (and, in the ablation, `impressions_90d`) let the tree take a shortcut through the label's own construction (`still_declining_may` has a floor effect when the late-window count is already near zero) — a leakage-*adjacent* problem, not a column that shouldn't have been there, but a label built in a way that a subset of legitimate features can trivially satisfy. Worth flagging as the more subtle lesson from this whole build: passing the "is this column from the future / from a product flag" checklist doesn't guarantee a feature can't still shortcut a poorly-constructed label.

In [10]:
# Confirm no banned columns made it into the final feature set
banned = {"content_updated_date", "last_optimized_date", "optimization_eligible_date",
          "is_published", "is_deleted", "impressions_may", "rate_may", "still_declining_may",
          "content_hash_id", "client_hash_id", "is_declining"}
final_features = set(numeric_features) | set(categorical_features)
leaked = final_features & banned
print(f"Final feature set: {sorted(final_features)}")
print(f"\nAny banned columns present? {bool(leaked)}  ({leaked if leaked else 'none'})")

Final feature set: ['avg_position_90d', 'char_count', 'clicks_90d', 'competition_level', 'content_type', 'ctr', 'imp_early', 'imp_late', 'impressions_90d', 'main_intent', 'search_volume', 'word_count']

Any banned columns present? False  (none)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

*(Pick your boldest sentence from `w04` or `w05` — a strong candidate is w05's Section 3 headline about the decision tree beating the baseline — and rewrite it here. Draft below, adjust to whichever sentence you actually want to soften.)*

**Before:** "The decision tree beats the baseline at every K, proving it's a better prioritization tool."

**After:** "Within this size-balanced client-grouped validation, the decision tree's precision@K was measured to be higher than the recomputed baseline rule at every K tested — but permutation importance showed this was driven almost entirely by a single feature tied mechanically to how the check-only label was constructed, not by a broader content-quality signal. The result is directional evidence that a learned model can outperform the hand-set rule on this exact label, and decision-support for further investigation with a better-constructed label — not proof the model captures real content decline.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.